In [3]:
import numpy as np
from math import log, sqrt, exp, factorial
from scipy.stats import norm, multivariate_normal

In [4]:
# =========================================================
# INPUTS — TU CHANGES JUSTE ICI AU QUIZ
# =========================================================

# Général
V0 = 100
V = 100

r = 0.03
sigma = 0.25

# Maturités
T = 2
t = 1

# Dettes
M = 80        # long terme
m = 50        # court terme

# Jumps (Q2)
lam = 0.4
mu_J = -0.1
sigma_J = 0.2

# Callable (Q3 & Q4)
p_call = 0.02

# Convertible (Q4)
conversion_ratio = 0.8

# Recovery (Q5)
theta = 1

# Binomial steps (NE PAS CHANGER)
N = 500

## Question 1 : MERTON (1974)

In [6]:
def bs_call(S, K, r, sigma, T):
    d1 = (log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * sqrt(T))
    d2 = d1 - sigma * sqrt(T)
    return S * norm.cdf(d1) - K * exp(-r * T) * norm.cdf(d2)


def merton_spread(V0, M, r, sigma, T):
    equity = bs_call(V0, M, r, sigma, T)
    debt = V0 - equity

    y = -(1 / T) * np.log(debt / M)
    spread = (y - r) * 10000

    return {
        "equity": equity,
        "debt": debt,
        "spread_bps": spread
    }


res_q1 = merton_spread(V0, M, r, sigma, T)
print(f"""
Q1 Results:
Equity: {res_q1['equity']:.2f}
Debt: {res_q1['debt']:.2f}
Spread: {res_q1['spread_bps']:.2f} bps
""")


Q1 Results:
Equity: 28.31
Debt: 71.69
Spread: 248.27 bps



## Question 2 : MERTON AVEC SAUTS (1976)

In [8]:
def merton_jump_call(V0, M, r, sigma, T, lam, mu_J, sigma_J, n_max=50):
    total = 0.0

    for n in range(n_max + 1):
        weight = exp(-lam * (1 + mu_J) * T) * ((lam * (1 + mu_J) * T) ** n) / factorial(n)

        r_n = r - lam * mu_J + n * log(1 + mu_J) / T
        sigma_n = sqrt(sigma**2 + n * sigma_J**2 / T)

        d1 = (log(V0 / M) + (r_n + 0.5 * sigma_n**2) * T) / (sigma_n * sqrt(T))
        d2 = d1 - sigma_n * sqrt(T)

        call = V0 * norm.cdf(d1) - M * exp(-r_n * T) * norm.cdf(d2)

        total += weight * call

    return total


def merton_jump_spread(V0, M, r, sigma, T, lam, mu_J, sigma_J):
    equity = merton_jump_call(V0, M, r, sigma, T, lam, mu_J, sigma_J)
    debt = V0 - equity

    y = -(1 / T) * np.log(debt / M)
    spread = (y - r) * 10000

    return {
        "equity": equity,
        "debt": debt,
        "spread_bps": spread
    }


res_q2 = merton_jump_spread(V0, M, r, sigma, T, lam, mu_J, sigma_J)
print(f"""
Q2 Results (Merton with Jumps):
Equity: {res_q2['equity']:.2f}
Debt: {res_q2['debt']:.2f}
Spread: {res_q2['spread_bps']:.2f} bps
""")


Q2 Results (Merton with Jumps):
Equity: 29.76
Debt: 70.24
Spread: 350.31 bps



## Question 3: ARBRE BINOMIAL CALLABLE

In [10]:
def callable_bond_binomial(V0, M, r, sigma, T, p_call, N=500):
    dt = T / N
    u = exp(sigma * sqrt(dt))
    d = 1 / u
    q = (exp(r * dt) - d) / (u - d)

    V = np.zeros((N + 1, N + 1))
    D = np.zeros((N + 1, N + 1))

    for i in range(N + 1):
        for j in range(i + 1):
            V[i, j] = V0 * (u ** j) * (d ** (i - j))

    for j in range(N + 1):
        D[N, j] = min(V[N, j], M)

    for i in range(N - 1, -1, -1):
        t_i = i * dt
        K_t = M * exp(-p_call * (T - t_i))

        for j in range(i + 1):
            continuation = exp(-r * dt) * (q * D[i + 1, j + 1] + (1 - q) * D[i + 1, j])

            D[i, j] = min(continuation, K_t)
            D[i, j] = min(D[i, j], V[i, j])

    price = D[0, 0]

    y = -(1 / T) * np.log(price / M)
    spread = (y - r) * 10000

    return {
        "price": price,
        "spread_bps": spread
    }


res_q3 = callable_bond_binomial(V0, M, r, sigma, T, p_call, N)
print(f"""
Q3 Results (Callable Bond - Binomial):
Price: {res_q3['price']:.2f}
Spread: {res_q3['spread_bps']:.2f} bps
""")


Q3 Results (Callable Bond - Binomial):
Price: 71.69
Spread: 248.06 bps



## QUESTION 4 — CALLABLE + CONVERTIBLE

In [12]:
def convertible_bond_binomial(V0, M, r, sigma, T, p_call, conversion_ratio, N=500):
    dt = T / N
    u = exp(sigma * sqrt(dt))
    d = 1 / u
    q = (exp(r * dt) - d) / (u - d)

    V = np.zeros((N + 1, N + 1))
    C = np.zeros((N + 1, N + 1))

    for i in range(N + 1):
        for j in range(i + 1):
            V[i, j] = V0 * (u ** j) * (d ** (i - j))

    for j in range(N + 1):
        debt_payoff = min(V[N, j], M)
        conversion = conversion_ratio * V[N, j]
        C[N, j] = max(debt_payoff, conversion)

    for i in range(N - 1, -1, -1):
        t_i = i * dt
        K_t = M * exp(-p_call * (T - t_i))

        for j in range(i + 1):
            continuation = exp(-r * dt) * (q * C[i + 1, j + 1] + (1 - q) * C[i + 1, j])

            conversion = conversion_ratio * V[i, j]
            after_call = max(conversion, K_t)

            C[i, j] = max(conversion, min(continuation, after_call))
            C[i, j] = min(C[i, j], V[i, j])

    price = C[0, 0]

    y = -(1 / T) * np.log(price / M)
    spread = (y - r) * 10000

    return {
        "price": price,
        "spread_bps": spread
    }


res_q4 = convertible_bond_binomial(V0, M, r, sigma, T, p_call, conversion_ratio, N)
print(f"""
Q4 Results (Callable + Convertible Bond):
Price: {res_q4['price']:.2f}
Spread: {res_q4['spread_bps']:.2f} bps
""")


Q4 Results (Callable + Convertible Bond):
Price: 80.00
Spread: -300.00 bps



## QUESTION 5 — DETTE COURT & LONG TERME

In [14]:
def Phi(x):
    return norm.cdf(x)

def Phi2(a, b, rho):
    mean = [0, 0]
    cov = [[1, rho], [rho, 1]]
    return multivariate_normal.cdf([a, b], mean=mean, cov=cov)


def compute_z(V, m, M, r, sigma, t, T):
    z1 = (log(V / m) + (r + 0.5 * sigma**2) * t) / (sigma * sqrt(t))
    z2 = (log(V / m) + (r - 0.5 * sigma**2) * t) / (sigma * sqrt(t))

    z3 = (log(V / M) + (r + 0.5 * sigma**2) * T) / (sigma * sqrt(T))
    z4 = (log(V / M) + (r - 0.5 * sigma**2) * T) / (sigma * sqrt(T))

    return z1, z2, z3, z4


def short_long_spreads(V, m, M, r, sigma, t, T, theta=1):
    z1, z2, z3, z4 = compute_z(V, m, M, r, sigma, t, T)
    rho = sqrt(t / T)

    d = (
        m * exp(-r * t) * Phi(z2)
        + (m / (m + M)) * theta * V * Phi(-z1)
    )

    D = (
        M * exp(-r * T) * Phi2(z2, z4, rho)
        + theta * V * Phi2(z1, -z3, -rho)
        + (M / (m + M)) * theta * V * Phi(-z1)
    )

    spread_short = (-(1 / t) * np.log(d / m) - r) * 10000
    spread_long = (-(1 / T) * np.log(D / M) - r) * 10000

    return {
        "short_value": d,
        "long_value": D,
        "short_spread_bps": spread_short,
        "long_spread_bps": spread_long
    }


res_q5 = short_long_spreads(V, m, M, r, sigma, t, T, theta)
print(f"""
Q5 Results (Short & Long-Term Debt):
Short-term value: {res_q5['short_value']:.2f}
Long-term value: {res_q5['long_value']:.2f}

Short spread: {res_q5['short_spread_bps']:.2f} bps
Long spread: {res_q5['long_spread_bps']:.2f} bps
""")


Q5 Results (Short & Long-Term Debt):
Short-term value: 48.43
Long-term value: 71.64

Short spread: 18.15 bps
Long spread: 251.66 bps



## Résumé des réponses

In [15]:
print("========== FINAL ANSWERS ==========\n")

# Q1
print(f"Q1: {res_q1['spread_bps']:.2f} bps")

# Q2
print(f"Q2: {res_q2['spread_bps']:.2f} bps")

# Q3
print(f"Q3: {res_q3['spread_bps']:.2f} bps")

# Q4
print(f"Q4: {res_q4['spread_bps']:.2f} bps")

# Q5
print(f"Q5 Short: {res_q5['short_spread_bps']:.2f} bps")
print(f"Q5 Long: {res_q5['long_spread_bps']:.2f} bps")

========== FINAL ANSWERS ==========

Q1: 248.27 bps
Q2: 350.31 bps
Q3: 248.06 bps
Q4: -300.00 bps
Q5 Short: 18.15 bps
Q5 Long: 251.66 bps
